In [2]:
from azureml.core import Workspace

ws = Workspace.from_config()
ws


Workspace.create(name='g10tfm-ml', subscription_id='506e4ca3-b9db-4960-aedf-c02bf0ea7e28', resource_group='jaragono-rg')

In [3]:
from azureml.core.environment import Environment

env = Environment.from_conda_specification(
    name="g10tfm-env",
    file_path="env.yml"
)


In [4]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(
    entry_script="score.py",
    environment=env
)


Warning, azureml-defaults not detected in provided environment pip dependencies. The azureml-defaults package contains requirements for the inference stack to run, and should be included.


In [5]:
from azureml.core.model import Model

registered_model = Model.register(
    workspace=ws,
    model_path="g10tfm_model_test1.pkl",
    model_name="g10tfm_model_test1"
)

registered_model


Registering model g10tfm_model_test1


Model(workspace=Workspace.create(name='g10tfm-ml', subscription_id='506e4ca3-b9db-4960-aedf-c02bf0ea7e28', resource_group='jaragono-rg'), name=g10tfm_model_test1, id=g10tfm_model_test1:2, version=2, tags={}, properties={})

In [6]:
from azureml.core.webservice import AciWebservice

aci_config = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    auth_enabled=True
)


In [9]:
service = Model.deploy(
    workspace=ws,
    name="g10tfm-credit-scoring-endpt-t1",
    models=[registered_model],
    inference_config=inference_config,
    deployment_config=aci_config
)

service.wait_for_deployment(show_output=True)


/tmp/ipykernel_3209/3573491983.py:1: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2025-12-01 06:11:55+00:00 Creating Container Registry if not exists.
2025-12-01 06:11:56+00:00 Building image..
2025-12-01 06:18:54+00:00 Generating deployment configuration..
2025-12-01 06:18:56+00:00 Submitting deployment to compute.
Failed


Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 0501a6d5-0a6e-4bd9-97d8-67e698718a8a
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '3e598908-4565-46da-b24d-20a30ef1b8a4' with object id '7f622c5e-bf9e-4ed3-a265-4f45d30561e5' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/506e4ca3-b9db-4960-aedf-c02bf0ea7e28/resourceGroups/jaragono-rg/providers/Microsoft.ContainerInstance/containerGroups/g10tfm-credit-scoring-endpt-t1-d6bSwxABfUS7tBUi3JF7yg' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}



WebserviceException: WebserviceException:
	Message: Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 0501a6d5-0a6e-4bd9-97d8-67e698718a8a
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '3e598908-4565-46da-b24d-20a30ef1b8a4' with object id '7f622c5e-bf9e-4ed3-a265-4f45d30561e5' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/506e4ca3-b9db-4960-aedf-c02bf0ea7e28/resourceGroups/jaragono-rg/providers/Microsoft.ContainerInstance/containerGroups/g10tfm-credit-scoring-endpt-t1-d6bSwxABfUS7tBUi3JF7yg' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Service deployment polling reached non-successful terminal st

In [10]:
print("Scoring URI:", service.scoring_uri)
print("Authentication Keys:", service.get_keys())


Scoring URI: None
Authentication Keys: ('Mr1wsExCc7xYLvqKcMLhjoOOezf7DtiW', 'NChsMoQXjYv0HNJwyfzBCntENUkNbrBQ')


In [11]:
import requests
import json

url = service.scoring_uri
key = service.get_keys()[0]

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}"
}

payload = {
    "features": [35, 800, 0.3, 72, 4, 200, 1, 300, 850, 1]
}

response = requests.post(url, json=payload, headers=headers)
print(response.json())


MissingSchema: Invalid URL 'None': No scheme supplied. Perhaps you meant https://None?